[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/Time-Series-Analysis/blob/main/EN/Course_Notebooks/chapter5b_multivariate_garch_lecture_notebook.ipynb)

---

# Multivariate GARCH Models: DCC, CCC, and BEKK

**Course:** Time Series Analysis and Forecasting  
**Program:** Bachelor program, Faculty of Cybernetics, Statistics and Economic Informatics, Bucharest University of Economic Studies, Romania  
**Academic Year:** 2025-2026

---

## Learning Objectives

By the end of this notebook, you will be able to:
1. Understand why multivariate volatility modeling is necessary
2. Compute and visualize time-varying correlations
3. Estimate DCC-GARCH models using the two-step procedure
4. Compare CCC vs DCC models and test for constant correlation
5. Apply multivariate GARCH for portfolio VaR and dynamic hedging

## Setup and Imports

In [ ]:
# Install packages if needed (for Colab)
try:
    from arch import arch_model
    import yfinance as yf
except ImportError:
    !pip install arch yfinance --quiet
    from arch import arch_model
    import yfinance as yf

# Core libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.optimize import minimize
import warnings
warnings.filterwarnings('ignore')

# Statistical models
from arch import arch_model
from statsmodels.stats.diagnostic import het_arch, acorr_ljungbox
from statsmodels.graphics.tsaplots import plot_acf

# Plotting style
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['font.size'] = 11
plt.rcParams['axes.facecolor'] = 'none'
plt.rcParams['figure.facecolor'] = 'none'
plt.rcParams['savefig.facecolor'] = 'none'
plt.rcParams['savefig.transparent'] = True
plt.rcParams['axes.grid'] = False
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['legend.frameon'] = False

# Colors (IDA scheme)
COLORS = {
    'blue': '#1A3A6E',
    'red': '#DC3545',
    'green': '#2E7D32',
    'orange': '#E67E22',
    'purple': '#8E44AD',
    'gray': '#666666'
}

print("All libraries loaded successfully!")

## 1. Why Multivariate GARCH?

**Univariate GARCH** models each asset's volatility in isolation. But financial decisions require:
- **Portfolio variance**: $\sigma_{p,t}^2 = \mathbf{w}'\mathbf{H}_t\mathbf{w}$ — needs the full covariance matrix
- **Dynamic hedging**: $h_t^* = h_{12,t}/h_{22,t}$ — needs time-varying covariances
- **Risk management**: correlations increase during crises — ignoring this underestimates risk

[QuantLet: TSA_ch5b_motivation](https://github.com/QuantLet/TSA/tree/main/TSA_ch5b/TSA_ch5b_motivation)

In [ ]:
# Download real data: S&P 500 and DAX
print("Downloading S&P 500 and DAX data from Yahoo Finance...")

sp500 = yf.download('^GSPC', start='2005-01-01', end='2024-12-31', progress=False)
dax = yf.download('^GDAXI', start='2005-01-01', end='2024-12-31', progress=False)

# Calculate returns
r_sp = (sp500['Close'].squeeze().pct_change() * 100).dropna()
r_dax = (dax['Close'].squeeze().pct_change() * 100).dropna()

# Align on common dates
returns = pd.concat([r_sp, r_dax], axis=1, join='inner')
returns.columns = ['SP500', 'DAX']
returns = returns.dropna()

print(f"\nBivariate Data Summary:")
print(f"Period: {returns.index[0].date()} to {returns.index[-1].date()}")
print(f"Observations: {len(returns)}")
print(f"\n{returns.describe().round(4)}")

In [ ]:
# Visualize: returns and rolling correlations
fig, axes = plt.subplots(3, 1, figsize=(14, 12), sharex=True)

# S&P 500 returns
axes[0].plot(returns.index, returns['SP500'], color=COLORS['blue'], linewidth=0.5, alpha=0.8)
axes[0].set_ylabel('Return (%)')
axes[0].set_title('S&P 500 Daily Returns', fontweight='bold')
axes[0].axhline(y=0, color='black', linewidth=0.5)

# DAX returns
axes[1].plot(returns.index, returns['DAX'], color=COLORS['red'], linewidth=0.5, alpha=0.8)
axes[1].set_ylabel('Return (%)')
axes[1].set_title('DAX Daily Returns', fontweight='bold')
axes[1].axhline(y=0, color='black', linewidth=0.5)

# Rolling correlation (60-day)
rolling_corr = returns['SP500'].rolling(60).corr(returns['DAX'])
axes[2].plot(returns.index, rolling_corr, color=COLORS['purple'], linewidth=0.8)
axes[2].axhline(y=rolling_corr.mean(), color=COLORS['gray'], linestyle='--', linewidth=1,
                label=f'Mean: {rolling_corr.mean():.2f}')
axes[2].set_ylabel('Correlation')
axes[2].set_xlabel('Date')
axes[2].set_title('60-Day Rolling Correlation: S&P 500 vs DAX', fontweight='bold')
axes[2].set_ylim(-0.2, 1.0)

# Mark crises
crisis_periods = [
    ('2008-09-01', '2009-03-31', '2008 Crisis'),
    ('2020-02-15', '2020-04-30', 'COVID-19'),
]
for start, end, label in crisis_periods:
    for ax in axes:
        ax.axvspan(pd.Timestamp(start), pd.Timestamp(end), alpha=0.15, color=COLORS['red'])

axes[2].legend(loc='upper center', bbox_to_anchor=(0.5, -0.15), ncol=2)
plt.tight_layout()
plt.show()

print("\nKey observation: Correlations spike during crises (2008, COVID) — they are NOT constant!")

## 2. The CCC Model (Bollerslev, 1990)

$$\mathbf{H}_t = \mathbf{D}_t \mathbf{R} \mathbf{D}_t$$

- $\mathbf{D}_t = \text{diag}(\sigma_{1,t}, \ldots, \sigma_{N,t})$ — univariate GARCH volatilities
- $\mathbf{R}$ — **constant** correlation matrix
- Each $\sigma_{i,t}^2$ follows GARCH(1,1)

**Two-step estimation:**
1. Estimate N univariate GARCH models → get standardized residuals $z_{it}$
2. Estimate $\hat{\mathbf{R}} = T^{-1}\sum_t \mathbf{z}_t\mathbf{z}_t'$

[QuantLet: TSA_ch5b_ccc_dcc](https://github.com/QuantLet/TSA/tree/main/TSA_ch5b/TSA_ch5b_ccc_dcc)

In [ ]:
# Step 1: Estimate univariate GARCH(1,1) for each series
print("CCC Step 1: Univariate GARCH(1,1) Estimation")
print("=" * 50)

# S&P 500
model_sp = arch_model(returns['SP500'].values, vol='Garch', p=1, q=1, dist='t')
res_sp = model_sp.fit(disp='off')
print(f"\nS&P 500: omega={res_sp.params['omega']:.6f}, alpha={res_sp.params['alpha[1]']:.4f}, "
      f"beta={res_sp.params['beta[1]']:.4f}, nu={res_sp.params['nu']:.2f}")

# DAX
model_dax = arch_model(returns['DAX'].values, vol='Garch', p=1, q=1, dist='t')
res_dax = model_dax.fit(disp='off')
print(f"DAX:     omega={res_dax.params['omega']:.6f}, alpha={res_dax.params['alpha[1]']:.4f}, "
      f"beta={res_dax.params['beta[1]']:.4f}, nu={res_dax.params['nu']:.2f}")

# Standardized residuals
z_sp = res_sp.std_resid
z_dax = res_dax.std_resid

# Step 2: Constant correlation from standardized residuals
R_ccc = np.corrcoef(z_sp, z_dax)
rho_ccc = R_ccc[0, 1]

print(f"\nCCC Step 2: Constant Correlation")
print(f"  rho_CCC = {rho_ccc:.4f}")
print(f"\n  This correlation is assumed CONSTANT for all t — is this realistic?")

## 3. The DCC Model (Engle, 2002)

$$\mathbf{H}_t = \mathbf{D}_t \mathbf{R}_t \mathbf{D}_t$$

where $\mathbf{R}_t$ is now **time-varying**:

$$\mathbf{Q}_t = (1-a-b)\bar{\mathbf{Q}} + a\,\mathbf{z}_{t-1}\mathbf{z}_{t-1}' + b\,\mathbf{Q}_{t-1}$$
$$\mathbf{R}_t = \text{diag}(\mathbf{Q}_t)^{-1/2}\,\mathbf{Q}_t\,\text{diag}(\mathbf{Q}_t)^{-1/2}$$

- $a$ = news reaction (how quickly correlations respond to shocks)
- $b$ = persistence (how slowly correlations revert to the mean)
- If $a = b = 0$ → DCC reduces to CCC

[QuantLet: TSA_ch5b_dcc_estimation](https://github.com/QuantLet/TSA/tree/main/TSA_ch5b/TSA_ch5b_dcc_estimation)

In [ ]:
# DCC Step 2: Estimate correlation dynamics
print("DCC Step 2: Estimating Dynamic Correlation Parameters (a, b)")
print("=" * 60)

# Stack standardized residuals
z = np.column_stack([z_sp, z_dax])
T, N = z.shape

# Unconditional correlation of standardized residuals
Qbar = z.T @ z / T

def dcc_loglik_and_corr(params, z, Qbar, return_corr=False):
    """DCC log-likelihood (correlation component only)."""
    a, b = params
    T, N = z.shape
    Qt = Qbar.copy()
    ll = 0.0
    rho_series = np.zeros(T)
    
    for t in range(1, T):
        Qt = (1 - a - b) * Qbar + a * np.outer(z[t-1], z[t-1]) + b * Qt
        # Rescale to correlation matrix
        d = np.sqrt(np.diag(Qt))
        Rt = Qt / np.outer(d, d)
        rho_series[t] = Rt[0, 1]
        
        # Log-likelihood contribution
        det_R = Rt[0,0]*Rt[1,1] - Rt[0,1]*Rt[1,0]
        Rt_inv = np.array([[Rt[1,1], -Rt[0,1]], [-Rt[1,0], Rt[0,0]]]) / det_R
        ll += np.log(det_R) + z[t] @ Rt_inv @ z[t] - z[t] @ z[t]
    
    if return_corr:
        return rho_series
    return 0.5 * ll

# Optimize
result = minimize(
    lambda p: dcc_loglik_and_corr(p, z, Qbar),
    x0=[0.02, 0.95],
    method='SLSQP',
    bounds=[(1e-6, 0.3), (1e-6, 0.9999)],
    constraints={'type': 'ineq', 'fun': lambda p: 0.9999 - p[0] - p[1]}
)

a_hat, b_hat = result.x
print(f"  a (news reaction):  {a_hat:.4f}")
print(f"  b (persistence):    {b_hat:.4f}")
print(f"  a + b:              {a_hat + b_hat:.4f}")
print(f"  Half-life:          {np.log(0.5)/np.log(a_hat + b_hat):.1f} days")

# Extract dynamic correlations
rho_dcc = dcc_loglik_and_corr([a_hat, b_hat], z, Qbar, return_corr=True)
rho_dcc[0] = rho_ccc  # Initialize

print(f"\n  Dynamic correlation range: [{rho_dcc[1:].min():.3f}, {rho_dcc[1:].max():.3f}]")
print(f"  Mean dynamic correlation: {rho_dcc[1:].mean():.3f}")

In [ ]:
# Visualize: DCC vs CCC vs Rolling Correlation
fig, axes = plt.subplots(2, 1, figsize=(14, 9))

# Panel 1: All three correlation measures
axes[0].plot(returns.index, rho_dcc, color=COLORS['blue'], linewidth=0.8, label='DCC')
axes[0].plot(returns.index, rolling_corr, color=COLORS['orange'], linewidth=0.6, alpha=0.6, label='Rolling 60d')
axes[0].axhline(y=rho_ccc, color=COLORS['red'], linestyle='--', linewidth=1.5,
                label=f'CCC (constant = {rho_ccc:.3f})')
axes[0].set_ylabel('Correlation')
axes[0].set_title('S&P 500 — DAX: Dynamic vs Constant Correlation', fontweight='bold')
axes[0].set_ylim(-0.1, 1.0)
axes[0].legend(loc='upper center', bbox_to_anchor=(0.5, -0.08), ncol=3)

# Panel 2: DCC deviation from CCC
deviation = rho_dcc - rho_ccc
axes[1].fill_between(returns.index, 0, deviation,
                     where=deviation > 0, color=COLORS['red'], alpha=0.4, label='Above CCC (higher risk)')
axes[1].fill_between(returns.index, 0, deviation,
                     where=deviation <= 0, color=COLORS['green'], alpha=0.4, label='Below CCC')
axes[1].axhline(y=0, color='black', linewidth=0.5)
axes[1].set_ylabel('DCC - CCC')
axes[1].set_xlabel('Date')
axes[1].set_title('DCC Deviation from Constant Correlation', fontweight='bold')
axes[1].legend(loc='upper center', bbox_to_anchor=(0.5, -0.15), ncol=2)

for start, end, label in crisis_periods:
    for ax in axes:
        ax.axvspan(pd.Timestamp(start), pd.Timestamp(end), alpha=0.1, color=COLORS['red'])

plt.tight_layout()
plt.show()

print("During 2008 crisis and COVID-19, DCC correlation spikes well above the CCC level.")
print("CCC underestimates risk precisely when it matters most!")

## 4. Testing CCC vs DCC: Engle-Sheppard Test

**$H_0$: Correlations are constant** (CCC is adequate)  
**$H_1$: Correlations vary over time** (need DCC)

Likelihood ratio test: DCC nests CCC (set $a = b = 0$)

[QuantLet: TSA_ch5b_ccc_test](https://github.com/QuantLet/TSA/tree/main/TSA_ch5b/TSA_ch5b_ccc_test)

In [ ]:
# Likelihood ratio test: CCC vs DCC
print("Likelihood Ratio Test: CCC vs DCC")
print("=" * 50)

# CCC log-likelihood (a=0, b=0)
ll_ccc = dcc_loglik_and_corr([0, 0], z, Qbar)
ll_dcc = result.fun  # DCC log-likelihood at optimum

LR = 2 * (ll_ccc - ll_dcc)  # Note: we minimized, so ll_ccc > ll_dcc means DCC is better
pval = 1 - stats.chi2.cdf(LR, df=2)

print(f"  CCC log-likelihood (correlation part): {-ll_ccc:.2f}")
print(f"  DCC log-likelihood (correlation part): {-ll_dcc:.2f}")
print(f"  LR statistic: {LR:.2f}")
print(f"  p-value: {pval:.6f}")
print(f"  Critical value (chi2, df=2, 1%): {stats.chi2.ppf(0.99, 2):.2f}")
print(f"\n  Conclusion: {'CCC REJECTED — use DCC' if pval < 0.01 else 'Cannot reject CCC'}")

## 5. Building the Time-Varying Covariance Matrix $\mathbf{H}_t$

$$\mathbf{H}_t = \mathbf{D}_t \mathbf{R}_t \mathbf{D}_t$$

where $\mathbf{D}_t = \text{diag}(\sigma_{1,t}, \sigma_{2,t})$ from univariate GARCH.

[QuantLet: TSA_ch5b_cov_matrix](https://github.com/QuantLet/TSA/tree/main/TSA_ch5b/TSA_ch5b_cov_matrix)

In [ ]:
# Build the full time-varying covariance matrix
sigma_sp = res_sp.conditional_volatility / 100  # Convert to decimal
sigma_dax = res_dax.conditional_volatility / 100

# DCC covariance: h_12,t = rho_t * sigma_1,t * sigma_2,t
h_11 = sigma_sp ** 2
h_22 = sigma_dax ** 2
h_12_dcc = rho_dcc * sigma_sp * sigma_dax
h_12_ccc = rho_ccc * sigma_sp * sigma_dax

# Visualize conditional covariance
fig, axes = plt.subplots(3, 1, figsize=(14, 12), sharex=True)

axes[0].plot(returns.index, sigma_sp * 100, color=COLORS['blue'], linewidth=0.8)
axes[0].set_ylabel('Volatility (%)')
axes[0].set_title('S&P 500: GARCH(1,1) Conditional Volatility', fontweight='bold')

axes[1].plot(returns.index, sigma_dax * 100, color=COLORS['red'], linewidth=0.8)
axes[1].set_ylabel('Volatility (%)')
axes[1].set_title('DAX: GARCH(1,1) Conditional Volatility', fontweight='bold')

axes[2].plot(returns.index, h_12_dcc * 10000, color=COLORS['purple'], linewidth=0.8, label='DCC')
axes[2].plot(returns.index, h_12_ccc * 10000, color=COLORS['orange'], linewidth=0.8, alpha=0.6, label='CCC')
axes[2].set_ylabel('Covariance (×10⁴)')
axes[2].set_xlabel('Date')
axes[2].set_title('Conditional Covariance: DCC vs CCC', fontweight='bold')
axes[2].legend(loc='upper center', bbox_to_anchor=(0.5, -0.15), ncol=2)

for start, end, label in crisis_periods:
    for ax in axes:
        ax.axvspan(pd.Timestamp(start), pd.Timestamp(end), alpha=0.1, color=COLORS['red'])

plt.tight_layout()
plt.show()

print("DCC covariance spikes higher than CCC during crises because correlations increase.")

## 6. Application: Portfolio VaR with DCC

$$\sigma_{p,t}^2 = \mathbf{w}'\mathbf{H}_t\mathbf{w}$$
$$\text{VaR}_t^\alpha = z_\alpha \cdot \sigma_{p,t}$$

For equal-weighted portfolio ($w_1 = w_2 = 0.5$):
$$\sigma_{p,t}^2 = 0.25 h_{11,t} + 0.25 h_{22,t} + 0.5 h_{12,t}$$

[QuantLet: TSA_ch5b_var_portfolio](https://github.com/QuantLet/TSA/tree/main/TSA_ch5b/TSA_ch5b_var_portfolio)

In [ ]:
# Portfolio VaR: DCC vs CCC
w = np.array([0.5, 0.5])

# Portfolio variance
port_var_dcc = w[0]**2 * h_11 + w[1]**2 * h_22 + 2 * w[0] * w[1] * h_12_dcc
port_var_ccc = w[0]**2 * h_11 + w[1]**2 * h_22 + 2 * w[0] * w[1] * h_12_ccc

port_vol_dcc = np.sqrt(port_var_dcc) * 100  # In percent
port_vol_ccc = np.sqrt(port_var_ccc) * 100

# VaR 99%
z_99 = stats.norm.ppf(0.99)
portfolio_value = 1_000_000
VaR_dcc = z_99 * port_vol_dcc / 100 * portfolio_value
VaR_ccc = z_99 * port_vol_ccc / 100 * portfolio_value

# Visualize
fig, axes = plt.subplots(2, 1, figsize=(14, 9), sharex=True)

axes[0].plot(returns.index, port_vol_dcc, color=COLORS['blue'], linewidth=0.8, label='DCC')
axes[0].plot(returns.index, port_vol_ccc, color=COLORS['red'], linewidth=0.8, alpha=0.6, label='CCC')
axes[0].set_ylabel('Portfolio Volatility (%)')
axes[0].set_title('Equal-Weighted Portfolio Volatility: DCC vs CCC', fontweight='bold')
axes[0].legend(loc='upper center', bbox_to_anchor=(0.5, -0.05), ncol=2)

axes[1].plot(returns.index, VaR_dcc, color=COLORS['blue'], linewidth=0.8, label='DCC VaR 99%')
axes[1].plot(returns.index, VaR_ccc, color=COLORS['red'], linewidth=0.8, alpha=0.6, label='CCC VaR 99%')
axes[1].fill_between(returns.index, VaR_ccc, VaR_dcc,
                     where=VaR_dcc > VaR_ccc, alpha=0.3, color=COLORS['orange'],
                     label='DCC > CCC (underestimation zone)')
axes[1].set_ylabel('VaR (EUR)')
axes[1].set_xlabel('Date')
axes[1].set_title(f'Portfolio VaR (99%, 1-day) for EUR {portfolio_value:,.0f}', fontweight='bold')
axes[1].legend(loc='upper center', bbox_to_anchor=(0.5, -0.12), ncol=3)

for start, end, label in crisis_periods:
    for ax in axes:
        ax.axvspan(pd.Timestamp(start), pd.Timestamp(end), alpha=0.1, color=COLORS['red'])

plt.tight_layout()
plt.show()

# Statistics
crisis_mask = (returns.index >= '2008-09-01') & (returns.index <= '2009-03-31')
print(f"\nVaR Comparison during 2008 Crisis:")
print(f"  Mean DCC VaR: EUR {VaR_dcc[crisis_mask].mean():,.0f}")
print(f"  Mean CCC VaR: EUR {VaR_ccc[crisis_mask].mean():,.0f}")
print(f"  CCC underestimates by: {((VaR_dcc[crisis_mask].mean()/VaR_ccc[crisis_mask].mean()-1)*100):.1f}%")

## 7. Application: Dynamic Hedging

**Minimum-variance hedge ratio:**
$$h_t^* = \frac{h_{12,t}}{h_{22,t}} = \rho_t \cdot \frac{\sigma_{1,t}}{\sigma_{2,t}}$$

- CCC: $h_t^* = \rho \cdot \sigma_{1,t}/\sigma_{2,t}$ — changes only through variance ratio
- DCC: $h_t^* = \rho_t \cdot \sigma_{1,t}/\sigma_{2,t}$ — changes through both correlation AND variances

[QuantLet: TSA_ch5b_hedge_ratio](https://github.com/QuantLet/TSA/tree/main/TSA_ch5b/TSA_ch5b_hedge_ratio)

In [ ]:
# Dynamic hedge ratios
hedge_dcc = h_12_dcc / h_22  # DCC hedge ratio
hedge_ccc = h_12_ccc / h_22  # CCC hedge ratio

fig, axes = plt.subplots(2, 1, figsize=(14, 9), sharex=True)

# Hedge ratios
axes[0].plot(returns.index, hedge_dcc, color=COLORS['blue'], linewidth=0.8, label='DCC')
axes[0].plot(returns.index, hedge_ccc, color=COLORS['red'], linewidth=0.8, alpha=0.6, label='CCC')
axes[0].axhline(y=1, color=COLORS['gray'], linestyle=':', linewidth=1)
axes[0].set_ylabel('Hedge Ratio')
axes[0].set_title('Dynamic Hedge Ratio: S&P 500 hedged with DAX futures', fontweight='bold')
axes[0].legend(loc='upper center', bbox_to_anchor=(0.5, -0.05), ncol=2)

# Difference
hedge_diff = hedge_dcc - hedge_ccc
axes[1].fill_between(returns.index, 0, hedge_diff,
                     where=hedge_diff > 0, color=COLORS['red'], alpha=0.4, label='DCC > CCC')
axes[1].fill_between(returns.index, 0, hedge_diff,
                     where=hedge_diff <= 0, color=COLORS['green'], alpha=0.4, label='DCC < CCC')
axes[1].axhline(y=0, color='black', linewidth=0.5)
axes[1].set_ylabel('Hedge Ratio Difference')
axes[1].set_xlabel('Date')
axes[1].set_title('DCC − CCC Hedge Ratio Difference', fontweight='bold')
axes[1].legend(loc='upper center', bbox_to_anchor=(0.5, -0.15), ncol=2)

for start, end, label in crisis_periods:
    for ax in axes:
        ax.axvspan(pd.Timestamp(start), pd.Timestamp(end), alpha=0.1, color=COLORS['red'])

plt.tight_layout()
plt.show()

print("During crises, DCC produces higher hedge ratios because correlations increase.")
print("CCC under-hedges in volatile periods — residual exposure remains.")

## 8. Minimum Variance Portfolio Weights

$$\mathbf{w}_t^{\text{MV}} = \frac{\mathbf{H}_t^{-1}\mathbf{1}}{\mathbf{1}'\mathbf{H}_t^{-1}\mathbf{1}}$$

For 2 assets:
$$w_{1,t}^* = \frac{h_{22,t} - h_{12,t}}{h_{11,t} + h_{22,t} - 2h_{12,t}}$$

[QuantLet: TSA_ch5b_mvp](https://github.com/QuantLet/TSA/tree/main/TSA_ch5b/TSA_ch5b_mvp)

In [ ]:
# Minimum Variance Portfolio weights
w1_dcc = (h_22 - h_12_dcc) / (h_11 + h_22 - 2 * h_12_dcc)
w1_ccc = (h_22 - h_12_ccc) / (h_11 + h_22 - 2 * h_12_ccc)

# Clip to [0, 1] for long-only
w1_dcc_clipped = np.clip(w1_dcc, 0, 1)
w1_ccc_clipped = np.clip(w1_ccc, 0, 1)

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(returns.index, w1_dcc_clipped, color=COLORS['blue'], linewidth=0.8, label='DCC: weight on S&P 500')
ax.plot(returns.index, w1_ccc_clipped, color=COLORS['red'], linewidth=0.8, alpha=0.6, label='CCC: weight on S&P 500')
ax.axhline(y=0.5, color=COLORS['gray'], linestyle=':', linewidth=1, label='Equal weight')
ax.set_ylabel('Weight on S&P 500')
ax.set_xlabel('Date')
ax.set_title('Minimum Variance Portfolio: S&P 500 Weight (DCC vs CCC)', fontweight='bold')
ax.set_ylim(0, 1)
ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.12), ncol=3)

for start, end, label in crisis_periods:
    ax.axvspan(pd.Timestamp(start), pd.Timestamp(end), alpha=0.1, color=COLORS['red'])

plt.tight_layout()
plt.show()

print("DCC-based weights adjust more dynamically as correlations change.")
print("When correlation increases, diversification benefit decreases → weights shift.")

## 9. Adding a Third Asset: Multi-Asset DCC

The beauty of DCC: adding more assets only adds 3 GARCH parameters per asset, but the DCC parameters $(a, b)$ remain just 2.

[QuantLet: TSA_ch5b_multi_asset](https://github.com/QuantLet/TSA/tree/main/TSA_ch5b/TSA_ch5b_multi_asset)

In [ ]:
# Add a third asset: Gold
print("Downloading Gold data...")
gold = yf.download('GC=F', start='2005-01-01', end='2024-12-31', progress=False)
r_gold = (gold['Close'].squeeze().pct_change() * 100).dropna()

# Align all three
returns_3 = pd.concat([r_sp, r_dax, r_gold], axis=1, join='inner').dropna()
returns_3.columns = ['SP500', 'DAX', 'Gold']

print(f"\nTrivariate Data: {len(returns_3)} observations")
print(f"\nCorrelation Matrix (full sample):")
print(returns_3.corr().round(3))

# Rolling correlations
fig, ax = plt.subplots(figsize=(14, 5))
rc_sp_dax = returns_3['SP500'].rolling(60).corr(returns_3['DAX'])
rc_sp_gold = returns_3['SP500'].rolling(60).corr(returns_3['Gold'])
rc_dax_gold = returns_3['DAX'].rolling(60).corr(returns_3['Gold'])

ax.plot(returns_3.index, rc_sp_dax, color=COLORS['blue'], linewidth=0.8, label='S&P 500 — DAX')
ax.plot(returns_3.index, rc_sp_gold, color=COLORS['orange'], linewidth=0.8, label='S&P 500 — Gold')
ax.plot(returns_3.index, rc_dax_gold, color=COLORS['green'], linewidth=0.8, label='DAX — Gold')
ax.axhline(y=0, color='black', linewidth=0.5)
ax.set_ylabel('Correlation')
ax.set_xlabel('Date')
ax.set_title('60-Day Rolling Correlations: 3-Asset Portfolio', fontweight='bold')
ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.12), ncol=3)

plt.tight_layout()
plt.show()

print("\nGold has low/negative correlation with equities — useful for diversification!")
print("But this correlation is NOT constant — it changes with market conditions.")

## 10. Model Diagnostics

After fitting DCC, check:
1. **Standardized residuals** $z_{it}$ should have no autocorrelation
2. **Squared standardized residuals** $z_{it}^2$ should have no ARCH effects
3. **Cross-products** $z_{it}z_{jt}$ should have no serial correlation

[QuantLet: TSA_ch5b_diagnostics](https://github.com/QuantLet/TSA/tree/main/TSA_ch5b/TSA_ch5b_diagnostics)

In [ ]:
# Diagnostics on standardized residuals
print("DCC Model Diagnostics")
print("=" * 50)

for name, z_i in [('S&P 500', z_sp), ('DAX', z_dax)]:
    print(f"\n{name}:")
    # Ljung-Box on z_i
    lb = acorr_ljungbox(z_i, lags=10, return_df=True)
    print(f"  Ljung-Box z_t (lag 10): Q={lb['lb_stat'].iloc[-1]:.2f}, "
          f"p={lb['lb_pvalue'].iloc[-1]:.4f} {'OK' if lb['lb_pvalue'].iloc[-1] > 0.05 else 'FAIL'}")
    
    # Ljung-Box on z_i^2
    lb2 = acorr_ljungbox(z_i**2, lags=10, return_df=True)
    print(f"  Ljung-Box z_t^2 (lag 10): Q={lb2['lb_stat'].iloc[-1]:.2f}, "
          f"p={lb2['lb_pvalue'].iloc[-1]:.4f} {'OK' if lb2['lb_pvalue'].iloc[-1] > 0.05 else 'FAIL'}")
    
    # ARCH-LM
    lm_stat, lm_pval, _, _ = het_arch(z_i, nlags=5)
    print(f"  ARCH-LM (5 lags): LM={lm_stat:.2f}, p={lm_pval:.4f} "
          f"{'OK' if lm_pval > 0.05 else 'FAIL'}")

# Cross-correlation check
cross = z_sp * z_dax
lb_cross = acorr_ljungbox(cross, lags=10, return_df=True)
print(f"\nCross z_SP * z_DAX:")
print(f"  Ljung-Box (lag 10): Q={lb_cross['lb_stat'].iloc[-1]:.2f}, "
      f"p={lb_cross['lb_pvalue'].iloc[-1]:.4f} {'OK' if lb_cross['lb_pvalue'].iloc[-1] > 0.05 else 'FAIL'}")

## Summary

### Key Takeaways

1. **Correlations are NOT constant** — they spike during crises
2. **CCC** (Bollerslev 1990): simple, parsimonious, but unrealistic
3. **DCC** (Engle 2002): dynamic correlations with only 2 extra parameters — the workhorse model
4. **Two-step estimation**: (1) univariate GARCH → (2) DCC parameters
5. **Applications**: portfolio VaR, dynamic hedging, minimum variance portfolios
6. **CCC underestimates risk** during crises by 10-20%
7. **BEKK** captures volatility spillovers but is limited to small N

### Practical Workflow
1. Compute rolling correlations → visualize non-constancy
2. Estimate univariate GARCH for each asset
3. Estimate DCC parameters (a, b)
4. Test CCC vs DCC (likelihood ratio)
5. Build $\mathbf{H}_t$ and apply for VaR, hedging, portfolio optimization
6. Diagnose: check standardized residuals